# Data Dharma by Srikanth
## SQL ↔ PySpark Bridge — Part 1
### Foundations: SELECT, FILTER, DISTINCT, SORT & LIMIT

### What will you learn?

One dataset.
One requirement.
Two approaches — SQL and PySpark.
Same result.

Every section in this notebook follows the same pattern:
Business Requirement → SQL → Result → PySpark → Result → Mapping.

```
                 SAME DATA
                    |
          ---------------------
          |                   |
      SQL VIEW             DATAFRAME
       orders              orders_df
          |                   |
         SQL               PySpark
          \                   /
           \                 /
              SAME RESULT
```

The SQL view `orders` and the DataFrame `orders_df` point to the **same underlying data**.
SQL cells query `orders`. PySpark cells operate on `orders_df`.

**How the two are connected:**

```
              SAME DATA
                  |
            orders_df
         PySpark DataFrame
                  |
      createOrReplaceTempView()
                  |
               orders
          Temporary SQL View
```

This is **not** two separately maintained datasets — `orders` is created directly from `orders_df`.

### PySpark Vocabulary — 60 Seconds

Before we start, remember these simple terms:

- Module → A toolbox containing reusable code
  Example: pyspark.sql.types

- Class → A blueprint/type available inside a module
  Examples: StructType, StructField, IntegerType, StringType

- List → A Python collection of items written inside [ ]
  Example: ["Houston", "Dallas", "Austin"]

- Schema → Defines the structure of the data:
  column names, data types, and nullability

- DataFrame → Table-like structured data that we work with in PySpark
  Example: orders_df

- Method → An operation we ask an object/DataFrame to perform
  Examples: .select(), .filter(), .orderBy(), .limit(), .withColumn()



- pyspark.sql.types        → Module

- StructType               → Class

- StructType(...)          → Object

-  [101, "Houston", 750.50]                    → List

- orders_schema            → Schema (StructType object)

- orders_df                → DataFrame

- orders_df.select(...)    → DataFrame method

## SECTION 0 — Data Setup

A small, realistic retail **orders** dataset — created entirely in-memory.
No external files, no cloud storage, no Unity Catalog permissions required.

*Recording note: Run this setup quickly. The purpose of this episode is SQL ↔ PySpark translation, not manually typing the sample dataset.*

In [0]:
# from datetime is a module in Python and date is a class in the datetime module
from datetime import date

# from pyspark.sql.types is a module in Python and StructType is a class in the datetime module
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

# from pyspark.sql.functions is a module in Python
# col is a function in the pyspark.sql.functions module
# round is a function in the pyspark.sql.functions module
from pyspark.sql.functions import col, round as spark_round

# below is the schema for the orders table
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_amount", DoubleType(), False),
])

# below is the list with sample data
orders_data = [
    (1001, 501, date(2026, 1, 5),  "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 3, 250.00, 20.00),
    (1002, 502, date(2026, 1, 6),  "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  2, 150.00, 10.00),
    (1003, 503, date(2026, 1, 7),  "Austin",   "TX", "PENDING",   "PAYPAL",      1, 500.00,  0.00),
    (1004, 504, date(2026, 1, 8),  "Chicago",  "IL", "COMPLETED", "UPI",         5,  80.00, 15.00),
    (1005, 505, date(2026, 1, 9),  "New York", "NY", "CANCELLED", "CREDIT_CARD", 4, 120.00,  0.00),
    (1006, 501, date(2026, 1, 10), "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 2, 300.00, 25.00),
    (1007, 506, date(2026, 1, 11), "Dallas",   "TX", "RETURNED",  "DEBIT_CARD",  1, 200.00,  0.00),
    (1008, 507, date(2026, 1, 12), "Austin",   "TX", "COMPLETED", "PAYPAL",      3, 100.00, 10.00),
    (1009, 508, date(2026, 1, 13), "Chicago",  "IL", "PENDING",   "UPI",         2,  60.00,  5.00),
    (1010, 509, date(2026, 1, 14), "New York", "NY", "COMPLETED", "CREDIT_CARD", 6,  90.00, 20.00),
    (1011, 502, date(2026, 1, 15), "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  4, 175.00, 30.00),
    (1012, 510, date(2026, 1, 16), "Houston",  "TX", "COMPLETED", "PAYPAL",      1, 800.00, 50.00),
    (1013, 511, date(2026, 1, 17), "Austin",   "TX", "CANCELLED", "UPI",         2, 250.00,  0.00),
    (1014, 512, date(2026, 1, 18), "Chicago",  "IL", "COMPLETED", "CREDIT_CARD", 3, 220.00, 10.00),
    (1015, 513, date(2026, 1, 19), "New York", "NY", "RETURNED",  "DEBIT_CARD",  1, 400.00,  0.00),
    (1016, 503, date(2026, 1, 20), "Austin",   "TX", "COMPLETED", "PAYPAL",      5,  60.00,  5.00),
    (1017, 514, date(2026, 1, 21), "Houston",  "TX", "PENDING",   "CREDIT_CARD", 2, 175.00,  0.00),
    (1018, 505, date(2026, 1, 22), "New York", "NY", "COMPLETED", "UPI",         3, 210.00, 15.00),
]

# create a dataframe with the schema and data
orders_raw_df = spark.createDataFrame(orders_data, schema=orders_schema)

Display(orders_raw_df)

In [0]:
display(orders_raw_df)

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0
1003,503,2026-01-07,Austin,TX,PENDING,PAYPAL,1,500.0,0.0
1004,504,2026-01-08,Chicago,IL,COMPLETED,UPI,5,80.0,15.0
1005,505,2026-01-09,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0
1007,506,2026-01-11,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0
1008,507,2026-01-12,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0
1009,508,2026-01-13,Chicago,IL,PENDING,UPI,2,60.0,5.0
1010,509,2026-01-14,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0


**Derived business measure:**

```
order_amount = (quantity * unit_price) - discount_amount
```

This is computed once, here, so SQL and PySpark always use the SAME value.

In [0]:
# create a new column 'order_amount' by multiplying 'quantity' and 'unit_price' and subtracting 'discount_amount'
# round the result to 2 decimal places and cast it to decimal(10,2)

orders_df = orders_raw_df.withColumn(
    "order_amount",
    spark_round((col("quantity") * col("unit_price")) - col("discount_amount"), 2).cast("decimal(10,2)")
)

# display the dataframe
display(orders_df)

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,290.00
1003,503,2026-01-07,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,500.00
1004,504,2026-01-08,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,385.00
1005,505,2026-01-09,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,480.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1007,506,2026-01-11,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,200.00
1008,507,2026-01-12,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,290.00
1009,508,2026-01-13,Chicago,IL,PENDING,UPI,2,60.0,5.0,115.00
1010,509,2026-01-14,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,520.00



SELECT
    *,

    CAST(
        ROUND((quantity * unit_price) - discount_amount, 2)
        AS DECIMAL(10,2)
    ) AS order_amount
FROM orders_raw;


In [0]:
# create a temp view 'orders' from the dataframe
orders_df.createOrReplaceTempView("orders")

In [0]:
%sql
SELECT
    *,

    CAST(
        ROUND((quantity * unit_price) - discount_amount, 2)
        AS DECIMAL(10,2)
    ) AS order_amount
FROM orders;

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00,730.00
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,290.00,290.00
1003,503,2026-01-07,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,500.00,500.00
1004,504,2026-01-08,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,385.00,385.00
1005,505,2026-01-09,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,480.00,480.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00,575.00
1007,506,2026-01-11,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,200.00,200.00
1008,507,2026-01-12,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,290.00,290.00
1009,508,2026-01-13,Chicago,IL,PENDING,UPI,2,60.0,5.0,115.00,115.00
1010,509,2026-01-14,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,520.00,520.00


From this point onward, SQL and PySpark are reading the same data —
SQL through the view `orders`, PySpark through the DataFrame `orders_df`.

## SECTION 1 — SELECT

**Business Requirement:** Show only `order_id`, `city`, `order_status`, and `order_amount`.

### 🟨 SQL

In [0]:
%sql

-- Select specific columns from the orders table

SELECT
    order_id,
    city,
    order_status,
    order_amount
FROM orders;

order_id,city,order_status,order_amount
1001,Houston,COMPLETED,730.00
1002,Dallas,COMPLETED,290.00
1003,Austin,PENDING,500.00
1004,Chicago,COMPLETED,385.00
1005,New York,CANCELLED,480.00
1006,Houston,COMPLETED,575.00
1007,Dallas,RETURNED,200.00
1008,Austin,COMPLETED,290.00
1009,Chicago,PENDING,115.00
1010,New York,COMPLETED,520.00


### 🟦 PySpark

In [0]:
# Select specific columns from the orders DataFrame

result_df = orders_df.select(
    "order_id",
    "city",
    "order_status",
    "order_amount"
)

# display the dataframe result_df
display(result_df)

order_id,city,order_status,order_amount
1001,Houston,COMPLETED,730.00
1002,Dallas,COMPLETED,290.00
1003,Austin,PENDING,500.00
1004,Chicago,COMPLETED,385.00
1005,New York,CANCELLED,480.00
1006,Houston,COMPLETED,575.00
1007,Dallas,RETURNED,200.00
1008,Austin,COMPLETED,290.00
1009,Chicago,PENDING,115.00
1010,New York,COMPLETED,520.00


**Key Mapping**

| SQL | PySpark |
|---|---|
| `SELECT column1, column2` | `df.select("column1", "column2")` |

`SELECT` ↔ `select()`

## SECTION 2 — ALIAS

**Business Requirement:** Display `order_amount` as `sales_amount`.

### 🟨 SQL

In [0]:
%sql
-- Select columns and rename order_amount as sales_amount using an alias

SELECT
    order_id,
    city,
    order_amount AS sales_amount
FROM orders;

order_id,city,sales_amount
1001,Houston,730.00
1002,Dallas,290.00
1003,Austin,500.00
1004,Chicago,385.00
1005,New York,480.00
1006,Houston,575.00
1007,Dallas,200.00
1008,Austin,290.00
1009,Chicago,115.00
1010,New York,520.00


order_amount AS sales_amount

              ↑
            Alias

### 🟦 PySpark

In [0]:
# Select columns and rename order_amount as sales_amount using an alias

result_df = orders_df.select(
    "order_id",
    "city",
    col("order_amount").alias("sales_amount")
)

display(result_df)

order_id,city,sales_amount
1001,Houston,730.00
1002,Dallas,290.00
1003,Austin,500.00
1004,Chicago,385.00
1005,New York,480.00
1006,Houston,575.00
1007,Dallas,200.00
1008,Austin,290.00
1009,Chicago,115.00
1010,New York,520.00


SQL       → order_amount AS sales_amount

PySpark   → col("order_amount").alias("sales_amount")

**Key Mapping**

SQL `AS` ↔ PySpark `alias()`

## SECTION 3 — FILTER / WHERE

**Business Requirement:** Show only `COMPLETED` orders.

### 🟨 SQL

In [0]:
%sql
-- Filter orders to return only those with COMPLETED status

SELECT *
FROM orders
WHERE order_status = 'COMPLETED';

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,290.00
1004,504,2026-01-08,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,385.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1008,507,2026-01-12,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,290.00
1010,509,2026-01-14,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,520.00
1011,502,2026-01-15,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,670.00
1012,510,2026-01-16,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,750.00
1014,512,2026-01-18,Chicago,IL,COMPLETED,CREDIT_CARD,3,220.0,10.0,650.00
1016,503,2026-01-20,Austin,TX,COMPLETED,PAYPAL,5,60.0,5.0,295.00


### 🟦 PySpark

In [0]:
# Filter orders to return only those with COMPLETED status

result_df = orders_df.filter(col("order_status") == "COMPLETED")

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,290.00
1004,504,2026-01-08,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,385.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1008,507,2026-01-12,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,290.00
1010,509,2026-01-14,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,520.00
1011,502,2026-01-15,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,670.00
1012,510,2026-01-16,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,750.00
1014,512,2026-01-18,Chicago,IL,COMPLETED,CREDIT_CARD,3,220.0,10.0,650.00
1016,503,2026-01-20,Austin,TX,COMPLETED,PAYPAL,5,60.0,5.0,295.00


**Key Mapping**

SQL `WHERE` ↔ PySpark `filter()`

`df.where(...)` is also available — it behaves the same as `filter()`.

**Note on `SELECT *`:** in SQL, `SELECT * FROM orders` means all columns.
In PySpark, the DataFrame already represents its columns, so `display(orders_df)` shows all of them —
or you can write it explicitly as `orders_df.select("*")`.
Not every SQL clause has one single exact DataFrame equivalent; this is just the closest match for `*`.

## SECTION 4 — MULTIPLE CONDITIONS

**Business Requirement:** Show `COMPLETED` orders from Texas with `order_amount` greater than $500.

### 🟨 SQL

In [0]:
%sql
-- Filter orders using multiple conditions

SELECT *
FROM orders
WHERE order_status = 'COMPLETED'
  AND state = 'TX'
  AND order_amount > 500;

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1011,502,2026-01-15,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,670.00
1012,510,2026-01-16,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,750.00


### 🟦 PySpark

In [0]:
# Filter orders using multiple conditions

result_df = orders_df.filter(
    (col("order_status") == "COMPLETED") &
    (col("state") == "TX") &
    (col("order_amount") > 500)
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1011,502,2026-01-15,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,670.00
1012,510,2026-01-16,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,750.00


**Beginner trap:**

SQL:
`AND` / `OR`

PySpark Column expressions:
`&` / `|` — used as `(condition1) & (condition2)` and `(condition1) | (condition2)`.

Python's own operator precedence is why each comparison needs its own parentheses in
expressions like these — for beginner teaching, the safe rule is: **always parenthesize
each PySpark Column condition.**

Also: do not replace these Column-expression combinations with ordinary Python `and` / `or` —
they do not work the same way on Spark Columns.

**One more example — OR:** Show orders from Houston or Dallas.

### 🟨 SQL

In [0]:
%sql
-- Filter orders where the city is either Houston or Dallas

SELECT *
FROM orders
WHERE city = 'Houston' OR city = 'Dallas';

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,290.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1007,506,2026-01-11,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,200.00
1011,502,2026-01-15,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,670.00
1012,510,2026-01-16,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,750.00
1017,514,2026-01-21,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,350.00


### 🟦 PySpark

In [0]:
# Filter orders where the city is either Houston or Dallas

result_df = orders_df.filter(
    (col("city") == "Houston") | (col("city") == "Dallas")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,order_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,order_amount
1001,501,2026-01-05,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,730.00
1002,502,2026-01-06,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,290.00
1006,501,2026-01-10,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,575.00
1007,506,2026-01-11,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,200.00
1011,502,2026-01-15,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,670.00
1012,510,2026-01-16,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,750.00
1017,514,2026-01-21,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,350.00


## SECTION 5 — DISTINCT

**Business Requirement:** Which cities have orders?

### 🟨 SQL

In [0]:
%sql
-- Select unique cities and sort them in ascending order

SELECT DISTINCT city
FROM orders
ORDER BY city;

city
Austin
Chicago
Dallas
Houston
New York


### 🟦 PySpark

In [0]:
# Select unique cities and sort them in ascending order

result_df = (
    orders_df
    .select("city")
    .distinct()
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

city
Austin
Chicago
Dallas
Houston
New York


**Key Mapping**

`DISTINCT` ↔ `distinct()`

## SECTION 6 — SORTING

**Business Requirement:** Show the highest-value orders first.

### 🟨 SQL

In [0]:
%sql
-- Select specific columns and sort orders by order_amount in descending order

SELECT order_id, city, order_amount
FROM orders
ORDER BY order_amount DESC;

order_id,city,order_amount
1012,Houston,750.00
1001,Houston,730.00
1011,Dallas,670.00
1014,Chicago,650.00
1018,New York,615.00
1006,Houston,575.00
1010,New York,520.00
1003,Austin,500.00
1013,Austin,500.00
1005,New York,480.00


### 🟦 PySpark

In [0]:
# Select specific columns and sort orders by order_amount in descending order

result_df = (
    orders_df
    .select("order_id", "city", "order_amount")
    .orderBy(col("order_amount").desc())
)

# display data in the DataFrame result_df
display(result_df)

order_id,city,order_amount
1012,Houston,750.00
1001,Houston,730.00
1011,Dallas,670.00
1014,Chicago,650.00
1018,New York,615.00
1006,Houston,575.00
1010,New York,520.00
1003,Austin,500.00
1013,Austin,500.00
1005,New York,480.00


**Key Mapping**

`ORDER BY` ↔ `orderBy()`

Ascending is the default. It can also be made explicit with `col("order_amount").asc()`.

## SECTION 7 — LIMIT

**Business Requirement:** Show only the top 5 highest-value orders.

### 🟨 SQL

In [0]:
%sql
-- Select the top 5 orders with the highest order_amount

SELECT order_id, city, order_amount
FROM orders
ORDER BY order_amount DESC
LIMIT 5;

order_id,city,order_amount
1012,Houston,750.00
1001,Houston,730.00
1011,Dallas,670.00
1014,Chicago,650.00
1018,New York,615.00


### 🟦 PySpark

In [0]:
# Select the top 5 orders with the highest order_amount

result_df = (
    orders_df
    .select("order_id", "city", "order_amount")
    .orderBy(col("order_amount").desc())
    .limit(5)
)

# display data in the DataFrame result_df
display(result_df)

order_id,city,order_amount
1012,Houston,750.00
1001,Houston,730.00
1011,Dallas,670.00
1014,Chicago,650.00
1018,New York,615.00


**Key Mapping**

`LIMIT` ↔ `limit()`

If "top" or "highest/lowest" is part of the requirement — **sort first, then limit.**
`limit()` alone, without sorting, does not guarantee the highest-value rows.

## SECTION 8 — COMBINING CONCEPTS

**Business Requirement:** Find the top 5 highest-value `COMPLETED` orders from Texas.

### 🟨 SQL

In [0]:
%sql
-- Select the top 5 completed orders in Texas with the highest order_amount

SELECT order_id, city, order_status, order_amount
FROM orders
WHERE order_status = 'COMPLETED'
  AND state = 'TX'
ORDER BY order_amount DESC
LIMIT 5;

order_id,city,order_status,order_amount
1012,Houston,COMPLETED,750.00
1001,Houston,COMPLETED,730.00
1011,Dallas,COMPLETED,670.00
1006,Houston,COMPLETED,575.00
1016,Austin,COMPLETED,295.00


### 🟦 PySpark

In [0]:
# Select the top 5 completed orders in Texas with the highest order_amount

result_df = (
    orders_df
    .filter(
        (col("order_status") == "COMPLETED") &
        (col("state") == "TX")
    )
    .select("order_id", "city", "order_status", "order_amount")
    .orderBy(col("order_amount").desc())
    .limit(5)
)

# display data in the DataFrame result_df
display(result_df)

order_id,city,order_status,order_amount
1012,Houston,COMPLETED,750.00
1001,Houston,COMPLETED,730.00
1011,Dallas,COMPLETED,670.00
1006,Houston,COMPLETED,575.00
1016,Austin,COMPLETED,295.00


Same requirement. Same filter → select → sort → limit logic. Same result — expressed two ways.

## SECTION 9 

**Requirement:** Show the top 3 highest-value `COMPLETED` orders from Houston or Dallas.

**Required output:** `order_id`, `city`, `order_status`, `order_amount`



### 🟨 SQL SOLUTION

In [0]:
%sql
-- Select the top 3 completed orders from Houston or Dallas with the highest order_amount

SELECT order_id, city, order_status, order_amount
FROM orders
WHERE order_status = 'COMPLETED'
  AND (city = 'Houston' OR city = 'Dallas')
ORDER BY order_amount DESC
LIMIT 3;

order_id,city,order_status,order_amount
1012,Houston,COMPLETED,750.00
1001,Houston,COMPLETED,730.00
1011,Dallas,COMPLETED,670.00


### 🟦 PYSPARK SOLUTION

In [0]:
# Select the top 3 completed orders from Houston or Dallas with the highest order_amount

result_df = (
    orders_df
    .filter(
        (col("order_status") == "COMPLETED") &
        ((col("city") == "Houston") | (col("city") == "Dallas"))
    )
    .select("order_id", "city", "order_status", "order_amount")
    .orderBy(col("order_amount").desc())
    .limit(3)
)

# display data in the DataFrame result_df
display(result_df)

order_id,city,order_status,order_amount
1012,Houston,COMPLETED,750.00
1001,Houston,COMPLETED,730.00
1011,Dallas,COMPLETED,670.00


**RESULT:** Both queries return the same 3 orders, in the same order — highest `order_amount` first.

## Final Check — Did SQL and PySpark Return the Same Result?

We wrote the same business logic in both SQL and PySpark.

Now let's verify that they actually returned the same `order_id` values in the same order.

### What will we do?

1. Run the SQL version and store the result in `sql_result`
2. Run the PySpark version and store the result in `pyspark_result`
3. Extract the `order_id` values from each result into Python lists
4. Compare the two lists

For example:

SQL → `[105, 103, 101]`  
PySpark → `[105, 103, 101]`

If both lists are identical → `True` ✅

In [0]:
# Compare SQL and PySpark results to verify that both return the same order_ids in the same order

# Run the SQL query and store the result as a DataFrame
sql_result = spark.sql("""
    SELECT order_id, city, order_status, order_amount
    FROM orders
    WHERE order_status = 'COMPLETED'
      AND state = 'TX'
    ORDER BY order_amount DESC
    LIMIT 5
""")

# Apply the same logic using PySpark and store the result as a DataFrame
pyspark_result = (
    orders_df
    .filter(
        (col("order_status") == "COMPLETED") &
        (col("state") == "TX")
    )
    .select("order_id", "city", "order_status", "order_amount")
    .orderBy(col("order_amount").desc())
    .limit(5)
)

# Trigger execution with collect(), bring the result to the driver, and extract order_id values into a Python list
sql_ids = [row["order_id"] for row in sql_result.collect()]

# Trigger execution with collect(), bring the PySpark result to the driver, and extract order_id values into a Python list
pyspark_ids = [row["order_id"] for row in pyspark_result.collect()]

# Display both order_id lists and check whether they match in the same order
print("SQL order_ids:     ", sql_ids)
print("PySpark order_ids: ", pyspark_ids)
print("Same order_ids (order-sensitive):", sql_ids == pyspark_ids)

SQL order_ids:      [1012, 1001, 1011, 1006, 1016]
PySpark order_ids:  [1012, 1001, 1011, 1006, 1016]
Same order_ids (order-sensitive): True


### Understanding the Result

`.collect()` brings the result rows from Spark to Python.

`row["order_id"]` takes the `order_id` value from each row.

The `[ ... ]` creates a Python list of those values.

So we end up with two lists:

- `sql_ids` → order IDs returned by SQL
- `pyspark_ids` → order IDs returned by PySpark

Finally:

`sql_ids == pyspark_ids`

checks whether both lists contain the same values **in the same order**.

If the result is `True` ✅, our SQL and PySpark versions returned the same ordered `order_id` results.

## SECTION 10 — FINAL CHEAT SHEET

| SQL | PySpark |
|---|---|
| `SELECT` | `select()` |
| `AS` | `alias()` |
| `WHERE` | `filter()` / `where()` |
| `AND` | `&` |
| `OR` | `\|` |
| `DISTINCT` | `distinct()` |
| `ORDER BY` | `orderBy()` |
| `DESC` | `desc()` |
| `LIMIT` | `limit()` |

```
     SQL
      ↕
   PySpark
      ↓
Same Business Logic
```

## Part 1 complete.

**Coming next — Part 2: SQL ↔ PySpark Transformations & Data Cleaning**

- `CASE WHEN`
- `CAST`
- `LIKE` / `IN`
- NULL handling
- String functions
- Date functions